# Notebook 4 — Análisis de Features ML para Clasificación de Vulnerabilidades

## Objetivo
Analizar en profundidad las **14 características** extraídas del código Java y su capacidad discriminativa para clasificar código como SEGURO (0) o VULNERABLE (1).

Este notebook complementa el EDA (`01_EDA_Vulnerability_Dataset.ipynb`) con un enfoque orientado al modelo:
- Distribución de cada feature por clase
- Análisis de correlación entre features
- Features más discriminativas (taint-like analysis)
- Tokenización y patrones léxicos del código
- Métricas de calidad del modelo XGBoost

### Técnicas aplicadas
| Técnica | Descripción |
|---------|-------------|
| **Tokenización** | Conteo de tokens léxicos (`\w+`) para capturar la densidad de código |
| **Análisis léxico (AST-inspired)** | Profundidad de anidamiento de llaves como proxy del AST depth |
| **Taint-like analysis** | Rastreo de fuentes peligrosas (`executeQuery`, `Runtime`) sin sanitización (`PreparedStatement`) |
| **Gradient Boosting (XGBoost)** | Ensemble de árboles de decisión, adecuado para features de conteo |
| **Validación cruzada 5-fold** | Estimación robusta del rendimiento generalizable |

In [ ]:
import sys
import os
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc, ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score

# Agregar raíz del proyecto al path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from security_checker.feature_extractor import JavaCodeAnalyzer

print('✓ Librerías cargadas')
print(f'  Root del proyecto: {ROOT}')

## 1. Cargar dataset procesado

In [ ]:
DATASET_PATH = ROOT / 'data' / 'processed' / 'security_dataset.csv'

if not DATASET_PATH.exists():
    print(f'⚠ Dataset no encontrado en {DATASET_PATH}')
    print('Ejecuta primero notebooks/02_Data_Preprocessing.ipynb')
else:
    df = pd.read_csv(DATASET_PATH)
    print(f'✓ Dataset cargado: {df.shape[0]:,} filas')
    print(f'  Columnas: {list(df.columns)}')
    print()
    print('Distribución de clases:')
    print(df['label'].value_counts().rename({0:'SEGURO', 1:'VULNERABLE'}))

## 2. Extracción de features sobre una muestra representativa

Por performance, extraemos las 14 features sobre una muestra de 2,000 registros (1,000 por clase).

In [ ]:
SAMPLE_SIZE = 1000

df_seg  = df[df['label']==0].sample(SAMPLE_SIZE, random_state=42)
df_vuln = df[df['label']==1].sample(SAMPLE_SIZE, random_state=42)
sample  = pd.concat([df_seg, df_vuln]).reset_index(drop=True)

print(f'Muestra: {len(sample)} registros ({SAMPLE_SIZE} por clase)')

features_list = []
for code in sample['code']:
    try:
        feats = JavaCodeAnalyzer.extract_features(str(code))
        features_list.append(feats)
    except Exception:
        features_list.append(JavaCodeAnalyzer._get_empty_features())

feature_names = sorted(features_list[0].keys())
X_sample = pd.DataFrame(features_list)[feature_names]
y_sample  = sample['label'].values

X_sample['label'] = y_sample
print(f'\nFeatures extraídas: {len(feature_names)}')
print(feature_names)
X_sample.describe().round(3)

## 3. Distribución de cada feature: SEGURO vs VULNERABLE

Box plots para visualizar si cada feature discrimina entre las dos clases.

In [ ]:
FEATURE_COLS = feature_names
n = len(FEATURE_COLS)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    data_seg  = X_sample[X_sample['label']==0][feat]
    data_vuln = X_sample[X_sample['label']==1][feat]
    
    bp = ax.boxplot(
        [data_seg, data_vuln],
        labels=['SEGURO', 'VULNERABLE'],
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2)
    )
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_ylabel('Valor', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

# Ocultar ejes sobrantes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de Features: SEGURO vs VULNERABLE', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Interpretación: Si las cajas no se solapan, la feature discrimina bien entre clases.')

## 4. Análisis de Correlación entre Features

In [ ]:
corr = X_sample[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Correlación de Pearson')

ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_yticks(range(len(FEATURE_COLS)))
ax.set_xticklabels(FEATURE_COLS, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(FEATURE_COLS, fontsize=8)

for i in range(len(FEATURE_COLS)):
    for j in range(len(FEATURE_COLS)):
        val = corr.iloc[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6, color=color)

ax.set_title('Matriz de Correlación de Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Pares más correlacionados
corr_upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
corr_stack = corr_upper.stack().sort_values(ascending=False)
print('Top 5 pares MÁS correlacionados:')
print(corr_stack.head(5).round(3))
print('\nTop 5 pares MÁS INVERSAMENTE correlacionados:')
print(corr_stack.tail(5).round(3))

## 5. Features de Seguridad — Análisis de Contaminación (Taint Analysis simplificado)

El análisis de contaminación rastrea:
- **Fuentes contaminadas (sources):** `executeQuery`, `Runtime.getRuntime`, `System.exec` — entradas peligrosas
- **Sumideros seguros (sinks/sanitizers):** `PreparedStatement`, `escape`, `validate` — neutralizan la contaminación

Código VULNERABLE = tiene sources **sin** sanitizers correspondientes.

In [ ]:
SECURITY_FEATS = [
    'dangerous_function_count',
    'sql_pattern_count',
    'system_call_count',
    'reflection_count',
    'sanitization_count'
]

labels_map = {0: 'SEGURO', 1: 'VULNERABLE'}

fig, axes = plt.subplots(1, len(SECURITY_FEATS), figsize=(20, 5))
colores = {'SEGURO': '#2ecc71', 'VULNERABLE': '#e74c3c'}

for ax, feat in zip(axes, SECURITY_FEATS):
    for lbl, nombre in labels_map.items():
        vals = X_sample[X_sample['label']==lbl][feat]
        ax.hist(vals, bins=20, alpha=0.6, label=nombre, color=colores[nombre], edgecolor='black')
    
    media_seg  = X_sample[X_sample['label']==0][feat].mean()
    media_vuln = X_sample[X_sample['label']==1][feat].mean()
    ax.axvline(media_seg,  color='#27ae60', linestyle='--', linewidth=2)
    ax.axvline(media_vuln, color='#c0392b', linestyle='--', linewidth=2)
    
    ax.set_title(feat.replace('_', '\n'), fontsize=9, fontweight='bold')
    ax.set_xlabel('Conteo')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Features de Seguridad: Análisis Taint-like\n(líneas = medias por clase)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== MEDIAS POR CLASE (features de seguridad) ===')
tabla = X_sample.groupby('label')[SECURITY_FEATS].mean().rename(index=labels_map)
print(tabla.round(3))

## 6. Tokenización — Análisis léxico

La tokenización es la primera etapa del análisis: descomponer el código en tokens (`\w+`) para obtener métricas de densidad léxica.

In [ ]:
import re

# Ejemplo de tokenización en código vulnerable vs seguro
ejemplo_vuln = '''String query = "SELECT * FROM users WHERE id = " + userId;
Statement stmt = conn.createStatement();
ResultSet rs = stmt.executeQuery(query);'''

ejemplo_seg = '''PreparedStatement stmt = conn.prepareStatement(
    "SELECT * FROM users WHERE id = ?");
stmt.setString(1, userId);
ResultSet rs = stmt.executeQuery();'''

tokens_vuln = re.findall(r'\w+', ejemplo_vuln)
tokens_seg  = re.findall(r'\w+', ejemplo_seg)

print('=== EJEMPLO DE TOKENIZACIÓN ===')
print(f'\nCódigo VULNERABLE ({len(tokens_vuln)} tokens):')
print(' | '.join(tokens_vuln))

print(f'\nCódigo SEGURO ({len(tokens_seg)} tokens):')
print(' | '.join(tokens_seg))

# Features extraídas de cada uno
print('\n=== FEATURES EXTRAÍDAS ===')
feats_vuln = JavaCodeAnalyzer.extract_features(ejemplo_vuln)
feats_seg  = JavaCodeAnalyzer.extract_features(ejemplo_seg)

comparacion = pd.DataFrame({
    'VULNERABLE': feats_vuln,
    'SEGURO': feats_seg
}).T
print(comparacion[sorted(comparacion.columns)].round(3))

## 7. Cargar modelo y evaluar métricas completas

In [ ]:
MODEL_PATH   = ROOT / 'model' / 'classifier_model.joblib'
SCALER_PATH  = ROOT / 'model' / 'scaler.joblib'
FEATS_PATH   = ROOT / 'model' / 'feature_names.json'

model  = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
with open(FEATS_PATH) as f:
    feat_names = json.load(f)

print(f'✓ Modelo cargado: {type(model).__name__}')
print(f'✓ Features: {len(feat_names)}')

In [ ]:
# Usar la muestra para evaluación
X = X_sample[feat_names].values
y = y_sample

X_scaled = scaler.transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')

print('=' * 60)
print('MÉTRICAS DE EVALUACIÓN (muestra de prueba 20%)')
print('=' * 60)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print()
print('Reporte completo:')
print(classification_report(y_test, y_pred, target_names=['SEGURO', 'VULNERABLE']))

## 8. Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['SEGURO', 'VULNERABLE'])
disp.plot(ax=axes[0], colorbar=True, cmap='Blues')
axes[0].set_title('Matriz de Confusión', fontweight='bold', fontsize=13)

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('Curva ROC', fontweight='bold', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Verdaderos Positivos (TP):  {tp:>4}  — Vulnerables detectados correctamente')
print(f'Verdaderos Negativos (TN):  {tn:>4}  — Seguros clasificados correctamente')
print(f'Falsos Positivos (FP):      {fp:>4}  — Seguros marcados como vulnerables')
print(f'Falsos Negativos (FN):      {fn:>4}  — Vulnerables no detectados (críticos)')
print(f'\nAUC-ROC: {roc_auc:.4f}')

## 9. Importancia de Features (XGBoost)

XGBoost reporta qué features usó más para construir los árboles.

In [ ]:
importancias = model.feature_importances_
feat_imp = pd.Series(importancias, index=feat_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colores_imp = ['#e74c3c' if f in [
    'dangerous_function_count', 'sql_pattern_count',
    'system_call_count', 'reflection_count', 'sanitization_count'
] else '#3498db' for f in feat_imp.index]

bars = ax.barh(feat_imp.index, feat_imp.values, color=colores_imp, edgecolor='black', alpha=0.8)

for bar, val in zip(bars, feat_imp.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Importancia (ganancia)', fontsize=11)
ax.set_title('Importancia de Features — XGBoost', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Leyenda
from matplotlib.patches import Patch
leyenda = [
    Patch(facecolor='#e74c3c', label='Features de seguridad'),
    Patch(facecolor='#3498db', label='Features estructurales/léxicas')
]
ax.legend(handles=leyenda, loc='lower right')
plt.tight_layout()
plt.show()

print('\nTop 5 features más importantes:')
print(feat_imp.sort_values(ascending=False).head(5).round(4))

## 10. Resumen de Métricas del Modelo

Tabla consolidada para incluir en el README e informe.

In [ ]:
print('=' * 65)
print('RESUMEN FINAL DEL MODELO XGBoost')
print('=' * 65)
print(f'  Algoritmo          : XGBoost (Gradient Boosting)')
print(f'  Features           : {len(feat_names)} características')
print(f'  Clases             : SEGURO (0) / VULNERABLE (1)')
print(f'  Umbral decisión    : prob ≥ 0.35 → VULNERABLE')
print()
print(f'  Accuracy           : {acc*100:.2f}%')
print(f'  Precision (weighted): {prec:.4f}')
print(f'  Recall (weighted)  : {rec:.4f}')
print(f'  F1-Score (weighted): {f1:.4f}')
print(f'  AUC-ROC            : {roc_auc:.4f}')
print()
print(f'  Falsos Negativos   : {fn} (vulnerables no detectados)')
print(f'  Falsos Positivos   : {fp} (seguros marcados como vulnerables)')
print('=' * 65)
print()
print(f'Interpretación:')
print(f'  • Accuracy > 82%: {"✓ CUMPLE" if acc >= 0.82 else "✗ NO CUMPLE"} el requisito mínimo')
print(f'  • Recall alto en VULNERABLE significa que detecta la mayoría')
print(f'    de los casos reales de vulnerabilidad (bajo fn)')